[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://github.com/codespaces)


<!-- gen:values.yml#weeks.04.notebook-title|Debrief: pandas -->
# Week 4 Debrief: pandas
<!-- /gen -->

*Worked answers to [Quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest) 04: The Messy Merger. They cover both cleaning strategies with their numbers, the merge that invents revenue, and this week's [AI failure gallery](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-failure-gallery).*

Read this after you have attempted the [quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest). These are worked answers, not the only answers. If your approach differs and your checks confirm it, your answer counts.

Run the data-setup cell below first. It rebuilds the quest's files exactly from the same seed, so this notebook runs on its own.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
# Hartley & Voss: Two sales ledgers that disagree, plus a loyalty contact file.
import os

import numpy as np
import pandas as pd

DATA_DIR = "assets/data/week04"
os.makedirs(DATA_DIR, exist_ok=True)
rng = np.random.default_rng(5813)  # module seed: Do not change

first_names = ["Amara", "Ben", "Chloe", "Dev", "Elena", "Farid", "Grace", "Hugo",
               "Isla", "Jack", "Kavya", "Liam", "Mei", "Noah", "Olive", "Priya",
               "Rosa", "Sam", "Tara", "Umar"]
last_names = ["Adeyemi", "Brooks", "Chen", "Dawson", "Evans", "Fletcher", "Gupta",
              "Hughes", "Iqbal", "Jones", "Khan", "Lawson", "Murphy", "Novak",
              "O'Brien", "Patel", "Quinn", "Reid", "Singh", "Turner"]
customers = sorted({f"{f} {l}" for f, l in
                    zip(rng.choice(first_names, 60), rng.choice(last_names, 60))})[:40]

hartley_products = {"Table Lamp": 34.00, "Cushion": 12.50, "Throw": 24.00,
                    "Candle Set": 9.75, "Mirror": 45.00, "Vase": 16.50,
                    "Photo Frame": 8.25, "Storage Basket": 14.00}
voss_products = {"Stock Pot": 42.00, "Chef's Knife": 55.00, "Mixing Bowl": 11.50,
                 "Cast Iron Pan": 65.00, "Measuring Jug": 6.75, "Serving Board": 18.00,
                 "Whisk": 7.50, "Teapot": 22.50}

all_days = pd.date_range("2026-01-01", "2026-06-30", freq="D")

# --- Hartley & Co: Title Case columns, DD/MM/YYYY dates, a TOTAL footer row,
# --- and 18 Amount values that went missing in the export (Qty/Unit Price intact).
n_h = 430
h_dates = rng.choice(all_days, n_h)
h_items = rng.choice(list(hartley_products), n_h)
h_qty = rng.integers(1, 6, n_h)
h_price = np.array([hartley_products[p] for p in h_items])
hartley = pd.DataFrame({
    "Date": pd.Series(h_dates).dt.strftime("%d/%m/%Y"),
    "Store": rng.choice(["Egham", "Staines", "Windsor"], n_h),
    "Product": h_items,
    "Customer": rng.choice(customers, n_h),
    "Qty": h_qty,
    "Unit Price": h_price,
    "Amount": np.round(h_qty * h_price, 2),
}).sort_values("Date", key=lambda s: pd.to_datetime(s, format="%d/%m/%Y"),
               ignore_index=True)
hartley_control_total = hartley["Amount"].sum()          # the TRUE Hartley total
missing_idx = rng.choice(hartley.index, 18, replace=False)
hartley.loc[missing_idx, "Amount"] = np.nan              # export mangled these
footer = pd.DataFrame([{"Date": "TOTAL", "Store": "", "Product": "", "Customer": "",
                        "Qty": np.nan, "Unit Price": np.nan,
                        "Amount": round(hartley_control_total, 2)}])
pd.concat([hartley, footer], ignore_index=True).to_csv(
    f"{DATA_DIR}/hartley_sales.csv", index=False)

# --- Voss Retail: snake_case columns, ISO dates, no amount column,
# --- and 14 rows accidentally exported twice.
n_v = 380
v_dates = rng.choice(all_days, n_v)
v_items = rng.choice(list(voss_products), n_v)
voss = pd.DataFrame({
    "txn_date": pd.Series(v_dates).dt.strftime("%Y-%m-%d"),
    "branch": rng.choice(["Kingston", "Richmond", "Guildford"], n_v),
    "item": v_items,
    "client_name": rng.choice(customers, n_v),
    "units": rng.integers(1, 7, n_v),
    "price_gbp": [voss_products[p] for p in v_items],
})
dup_rows = voss.sample(14, random_state=5813)
voss = (pd.concat([voss, dup_rows])
        .sample(frac=1, random_state=5813)     # shuffle the duplicates in
        .sort_values("txn_date", ignore_index=True))
voss.to_csv(f"{DATA_DIR}/voss_ledger.csv", index=False)

# --- Loyalty contacts: One row per member, except two names that belong
# --- to TWO different people each (same name, different member id and tier).
loyalty = pd.DataFrame({
    "member_id": [f"M{1000 + i}" for i in range(len(customers))],
    "customer_name": customers,
    "loyalty_tier": rng.choice(["Bronze", "Silver", "Gold"], len(customers)),
})
namesakes = pd.DataFrame({
    "member_id": ["M9001", "M9002"],
    "customer_name": [customers[7], customers[21]],   # second person, same name
    "loyalty_tier": ["Gold", "Gold"],
})
loyalty = pd.concat([loyalty, namesakes], ignore_index=True)
loyalty.to_csv(f"{DATA_DIR}/loyalty_contacts.csv", index=False)

for f in ["hartley_sales.csv", "voss_ledger.csv", "loyalty_contacts.csv"]:
    print(f"Created {DATA_DIR}/{f}")


## 🔍 First look: The surprises

1. **The schemas disagree**: Hartley has 7 columns and Voss has 6. They share no column names, and Voss has no amount column
2. **Hartley's `TOTAL` footer**: The last row is not a sale. It is the ledger's own grand total (£26,491.25)
3. **Missing values and duplicates**: Hartley has 18 missing `Amount` values, and Voss has 14 exact duplicate rows. Hartley's `Qty` is a decimal column (`float64`) because the footer's `Qty` cell is empty
4. **Hartley's dates fail to parse**: Without `format=`, `pd.to_datetime` guesses month-first from rows like `01/01/2026`. It then reaches `13/01/2026` and raises an error. The <!-- plain-language: everyday Gallery -->gallery below shows the version that fails with no error

## 🥉 Bronze: Worked answer

The answer standardises each ledger to the shared schema, stacks the two, and groups by month. Each cleaning step prints a count of what it changed. Those counts are what the Partner meant by "combine them like someone who will be asked questions afterwards".

In [ ]:
import pandas as pd

h = pd.read_csv("assets/data/week04/hartley_sales.csv")
v = pd.read_csv("assets/data/week04/voss_ledger.csv")

# --- Hartley: Footer out, dates parsed, schema standardised ---
hartley_clean = h[h["Date"] != "TOTAL"].copy()
print(f"Hartley: {len(h)} rows -> {len(hartley_clean)} after removing footer")

hartley_clean["date"] = pd.to_datetime(hartley_clean["Date"], format="%d/%m/%Y")
# Bronze choice: Repair missing Amounts from Qty x Unit Price (Silver weighs this)
n_missing = hartley_clean["Amount"].isna().sum()
hartley_clean["amount"] = hartley_clean["Amount"].fillna(
    hartley_clean["Qty"] * hartley_clean["Unit Price"])
print(f"Hartley: repaired {n_missing} missing Amount values")

hartley_clean = hartley_clean.rename(columns={
    "Store": "store", "Product": "product", "Customer": "customer"})
hartley_clean["company"] = "Hartley"
hartley_clean = hartley_clean[["date", "store", "product", "customer",
                               "amount", "company"]]

The cell below cleans Voss the same way. Voss has duplicates instead of a footer, and its amount column is computed rather than repaired.

In [ ]:
# --- Voss: Duplicates out, dates parsed, amount computed, schema standardised ---
n_dups = v.duplicated().sum()
voss_clean = v.drop_duplicates().copy()
print(f"Voss: {len(v)} rows -> {len(voss_clean)} after dropping {n_dups} duplicates")

voss_clean["date"] = pd.to_datetime(voss_clean["txn_date"], format="%Y-%m-%d")
voss_clean["amount"] = voss_clean["units"] * voss_clean["price_gbp"]
voss_clean = voss_clean.rename(columns={
    "branch": "store", "item": "product", "client_name": "customer"})
voss_clean["company"] = "Voss"
voss_clean = voss_clean[["date", "store", "product", "customer",
                         "amount", "company"]]

Stack the two ledgers, then print the monthly table.

In [ ]:
# --- Combine and report ---
combined = pd.concat([hartley_clean, voss_clean], ignore_index=True)
print(f"Combined: {len(hartley_clean)} + {len(voss_clean)} = {len(combined)} rows\n")

monthly = combined.groupby(combined["date"].dt.to_period("M"))["amount"].sum()
print("Hartley & Voss, combined monthly revenue, H1 2026:")
for month, rev in monthly.items():
    print(f"  {month}:  £{rev:>10,.2f}")
print(f"  H1 total: £{monthly.sum():,.2f}")

**The check that earns [Bronze](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bronze)**: The cell below adds up February by hand and compares it with the table. It also checks that the row counts add up.

In [ ]:
# Hand-check February: Filter each cleaned ledger, sum, add. Does it match the table?
feb_h = hartley_clean[hartley_clean["date"].dt.month == 2]["amount"].sum()
feb_v = voss_clean[voss_clean["date"].dt.month == 2]["amount"].sum()
feb_table = monthly[monthly.index.month == 2].iloc[0]
print(f"Feb, by hand:  £{feb_h:,.2f} + £{feb_v:,.2f} = £{feb_h + feb_v:,.2f}")
print(f"Feb, in table: £{feb_table:,.2f}")
assert abs((feb_h + feb_v) - feb_table) < 0.01, "February does not reconcile!"

# Row arithmetic: Everything in must be accounted for out
print(f"\nRows: Hartley {len(h)} - 1 footer = {len(hartley_clean)};")
print(f"      Voss {len(v)} - {n_dups} dups = {len(voss_clean)};")
print(f"      combined = {len(combined)} (= {len(hartley_clean)} + {len(voss_clean)})")

## 🥈 Silver: The evidence, and a defence

Strategy A (drop) and Strategy B (repair and flag) differ **only** in Hartley's 18 missing-`Amount` rows. The cell below counts the rows, totals the pounds, and shows which months they fall in.

In [ ]:
# What separates Strategy A (drop) from Strategy B (repair + flag)?
h_body = h[h["Date"] != "TOTAL"].copy()
suspect = h_body[h_body["Amount"].isna()]
stake = (suspect["Qty"] * suspect["Unit Price"]).sum()

print(f"Rows at stake:   {len(suspect)} of {len(h_body)} Hartley sales "
      f"({len(suspect) / len(h_body):.1%})")
print(f"Pounds at stake: £{stake:,.2f}")

# Where would the gap land? Month by month:
suspect_dates = pd.to_datetime(suspect["Date"], format="%d/%m/%Y")
gap_by_month = (suspect["Qty"] * suspect["Unit Price"]).groupby(
    suspect_dates.dt.to_period("M")).sum()
print("\nStrategy A would under-report, by month:")
print(gap_by_month.round(2))

Next, check the repair itself. A repair formula is only defensible if it reproduces the values that survived the export.

In [ ]:
# The trust check: Is Qty x Unit Price a faithful stand-in for Amount?
# Test it on the rows where Amount SURVIVED:
intact = h_body.dropna(subset=["Amount"])
recomputed = (intact["Qty"] * intact["Unit Price"]).round(2)
agree = (recomputed == intact["Amount"].round(2)).mean()
print(f"Qty x Unit Price reproduces Amount on {agree:.1%} of intact rows")

Here is **one defence**. Yours may differ and still be right. What earns [Silver](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-silver) is the evidence.

> I recommend **Strategy B (repair and flag)**. The 18 suspect rows carry £1,193.75 of real sales (about 4% of Hartley's sales). Dropping them would under-report H1 revenue, with the gap scattered across several months. The repair is not an estimate: `Qty × Unit Price` reproduces `Amount` exactly on every intact row. The `repaired` flag keeps the <!-- plain-language: everyday decision -->decision auditable. The repaired ledger also reconciles with Hartley's own control total to the penny ([Gold](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-gold) confirms this). The dropped version cannot.

**When A wins instead:** Strategy A, drop and disclose, is the defensible choice in these cases:

- The surviving columns are suspect too, so there is no way to check the repair
- The missing rows are a negligible share of the ledger, not 4%
- The client's audit rules forbid filled-in values

Either strategy can be right. What matters is that you show the pounds and disclose the <!-- plain-language: everyday decision -->decision.

## 🥇 Gold: Worked sketches

### Sketch 1: Reconcile against the control total

The footer row you removed in Bronze is also the strongest evidence in the file. Hartley's own system computed it **before** the export lost the 18 amounts. The cell below reads that control total from the footer and tests both strategies against it.

In [ ]:
# Extract the control BEFORE dropping the footer, then test both strategies
control_total = h.loc[h["Date"] == "TOTAL", "Amount"].iloc[0]

total_a = h_body["Amount"].dropna().sum()                       # Strategy A
total_b = h_body["Amount"].fillna(h_body["Qty"] * h_body["Unit Price"]).sum()  # B

print(f"Hartley control total (from the footer): £{control_total:,.2f}")
print(f"Strategy A (drop):    £{total_a:,.2f}   gap: £{control_total - total_a:,.2f}")
print(f"Strategy B (repair):  £{total_b:,.2f}   gap: £{control_total - total_b:,.2f}")

Strategy B reconciles to the penny. Strategy A is £1,193.75 short, exactly the pounds at stake in Silver. A control total is an independent number your cleaning has to match. Real ledgers usually have one, such as a system total, a bank statement, or last year's audited accounts. Ask for it before you clean.

### Sketch 2: A reusable data-quality report

Run one function on every raw export, before anyone cleans anything. The cell below defines it and runs it on both raw ledgers.

In [ ]:
def quality_report(df, name, amount_col=None):
    """One-screen data-quality summary for a raw ledger export."""
    print(f"=== {name}: {len(df)} rows x {len(df.columns)} columns ===")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")
    missing = df.isna().sum()
    missing = missing[missing > 0]
    print(f"Missing values: {missing.to_dict() if len(missing) else 'none'}")
    if amount_col is not None:
        amounts = pd.to_numeric(df[amount_col], errors="coerce")
        bad = (amounts <= 0).sum()
        print(f"Zero/negative {amount_col}: {bad}")
        print(f"Largest {amount_col}: {amounts.max():,.2f} (a footer-sized outlier?)")
    print()

quality_report(h, "Hartley raw", amount_col="Amount")
quality_report(v, "Voss raw")

On raw Hartley, the report shows a maximum `Amount` hundreds of times larger than a normal sale. That is the footer, found from its size alone. Next month's export can go through the same function with no new code. That reuse is what earns Gold.

## 🚩 Red Flags: Worked answer

**The bug:** The draft joined sales to `loyalty_contacts.csv` **on customer name**, and names are not unique. The loyalty file contains two pairs of namesakes: Two members are called *Dev Gupta* and two are called *Jack Chen*. The two members in each pair have different `member_id` values. A merge repeats every sales row once per match, so every sale by those names counts **twice**.

The cell below rebuilds the draft's inputs and measures the damage in pounds.

In [ ]:
# Rebuild the draft's inputs and quantify the damage
loyalty = pd.read_csv("assets/data/week04/loyalty_contacts.csv")

with_tiers = combined.merge(loyalty, left_on="customer",
                            right_on="customer_name", how="left")

dup_names = loyalty.loc[loyalty["customer_name"].duplicated(), "customer_name"]
phantom = with_tiers["amount"].sum() - combined["amount"].sum()
affected = combined[combined["customer"].isin(dup_names)]

print(f"Duplicated names in loyalty file: {list(dup_names)}")
print(f"Rows: {len(combined)} before merge -> {len(with_tiers)} after "
      f"(+{len(with_tiers) - len(combined)})")
print(f"True H1 total:    £{combined['amount'].sum():,.2f}")
print(f"Draft's total:    £{with_tiers['amount'].sum():,.2f}")
print(f"Phantom revenue:  £{phantom:,.2f} "
      f"({len(affected)} sales rows double-counted)")

**In business terms:** The draft reports £2,635.25 of revenue that does not exist. The merge credits all of it to the loyalty tiers the namesakes belong to. The board would then set each tier's marketing budget from an inflated table. The numbers would fail the first check Finance runs.

**The catch, in general:** A `how="left"` merge must never add rows to the table on the left. Assert `len(before) == len(after)` every time.

**A defensible fix:** The sales ledgers carry no member ID, so this data cannot tell the namesakes apart. Say so rather than guess. The cell below joins only on names that are unique in the loyalty file and reports the rest as unresolved.

In [ ]:
# Fix: Join on unambiguous names only; report namesakes as unresolved
ambiguous = loyalty.loc[loyalty["customer_name"].duplicated(keep=False),
                        "customer_name"].unique()
loyalty_unique = loyalty[~loyalty["customer_name"].isin(ambiguous)]

fixed = combined.merge(loyalty_unique, left_on="customer",
                       right_on="customer_name", how="left")
fixed["loyalty_tier"] = fixed["loyalty_tier"].fillna("Unresolved (namesake)")

assert len(fixed) == len(combined), "merge must not add rows"
assert abs(fixed["amount"].sum() - combined["amount"].sum()) < 0.01

print(fixed.groupby("loyalty_tier")["amount"].sum().round(2))
print(f"\nGrand total: £{fixed['amount'].sum():,.2f}  (matches the trusted total)")

A fix that asks Finance for a `member_id` on the sales ledger is also right. What earns the [badge](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-badge) is that the total is correct and you can prove it.

## 🤖 AI failure gallery

Each entry below shows a way an AI tool can get this week's material wrong.

### 1. A footer row read as a sale

**Plausible [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate) output:** Asked for "total revenue from hartley_sales.csv", it writes `read_csv` then `.sum()` and reports the figure with confidence. The cell below prints that figure next to the true total.

In [ ]:
# The Associate's 'total revenue'. It runs clean and looks authoritative
h_raw = pd.read_csv("assets/data/week04/hartley_sales.csv")
naive_total = h_raw["Amount"].sum()
true_total = h_raw[h_raw["Date"] != "TOTAL"]["Amount"].fillna(
    h_raw["Qty"] * h_raw["Unit Price"]).sum()

print(f"Associate reports H1 Hartley revenue: £{naive_total:,.2f}")
print(f"Actual H1 Hartley revenue:            £{true_total:,.2f}")

**Why it is wrong:** `read_csv` cannot know that the last row is a grand-total footer, so the ledger's own sum is added to itself. Revenue comes out at almost exactly **double**. The number looks plausible, so it reaches the board pack unchecked.

**The catch:** Run `df.tail()` on any file you did not create. If `df["Amount"].max()` returns a value hundreds of times the typical sale, that is the footer. Remove it with `skipfooter=1`, or filter it out.

### 2. Revenue read as text

**Plausible Associate output:** A finance system exports amounts with thousands separators (`"1,234.56"`). The Associate loads the file. The table displays perfectly, so the Associate carries on to the analysis. The cell below imitates the export with [`io.StringIO`](https://docs.python.org/3/library/io.html#io.StringIO), which lets a string stand in for a file.

In [ ]:
import io

csv_text = "month,amount\nJan,\"12,450.00\"\nFeb,\"9,880.50\"\nMar,\"11,020.25\"\n"
df_coerced = pd.read_csv(io.StringIO(csv_text))

print(df_coerced)                    # looks perfect...
print(f"\ndtype of amount: {df_coerced['amount'].dtype}")   # ...but it is TEXT
print(f"'sum' of the column: {df_coerced['amount'].sum()!r}")

**Why it is wrong:** The commas make pandas read `amount` as text, so "summing" the column joins the strings into one meaningless value. Sorting and `.max()` go wrong **without any error**: Text sorts character by character, so `.max()` returns `"9,880.50"`, not `"12,450.00"`.

**The catch:** Check `.dtypes` (the type pandas gave each column) immediately after every read. A revenue column that is not a number is a warning sign. **The fix:** [`pd.read_csv(..., thousands=",")`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html), or `pd.to_numeric` after stripping the commas.

### 3. The merge default that deletes sales

**Plausible Associate output:** Asked to "add product details to the sales table", it writes `pd.merge(sales, products, on="product")` with no `how=`. The result *looks* complete. The cell below merges five sales with a product list that is missing one product.

In [ ]:
sales_g = pd.DataFrame({"product": ["Lamp", "Vase", "Teapot", "Whisk", "Lamp"],
                        "amount": [34.0, 16.5, 22.5, 7.5, 34.0]})
# The product reference is missing 'Whisk' (new line, not catalogued yet)
products_g = pd.DataFrame({"product": ["Lamp", "Vase", "Teapot"],
                           "category": ["Lighting", "Decor", "Kitchen"]})

merged_g = pd.merge(sales_g, products_g, on="product")   # how defaults to INNER
print(merged_g)
print(f"\nSales before merge: £{sales_g['amount'].sum():,.2f} ({len(sales_g)} rows)")
print(f"Sales after merge:  £{merged_g['amount'].sum():,.2f} ({len(merged_g)} rows)")

**Why it is wrong:** `pd.merge` defaults to `how="inner"`, which keeps only matched rows. The Whisk sale is not marked missing. It is **gone**, so reported revenue shrinks with no warning. The draft answers "revenue for catalogued products", not "revenue".

**The catch:** Count the rows before and after every merge. Add `indicator=True` while you work: The `left_only` rows are the money a default `inner` merge drops. **The fix:** Use `how="left"` when you add columns to a main table. Then decide what to do with the unmatched rows.

### 4. Dates in the wrong month

In [First look](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-first-look), `pd.to_datetime` without a format **raises an error** because day 13 exists in the data. The dangerous case is a small file where every day number is 12 or below. The dates then parse month-first, with no error. The cell below parses three UK dates with no format given.

In [ ]:
# Three UK-format transactions (5 Jan, 3 Feb, 10 Mar): All days <= 12
uk_dates = pd.Series(["05/01/2026", "03/02/2026", "10/03/2026"])
parsed = pd.to_datetime(uk_dates)          # no format given -> pandas guesses

for raw, p in zip(uk_dates, parsed):
    print(f"{raw}  ->  {p:%d %B %Y}")

**Why it is wrong:** 5 January becomes 1 May, 3 February becomes 2 March, and 10 March becomes 3 October. Every transaction moves to the wrong month, so monthly revenue, seasonality, and any "Q1 vs Q2" comparison are all incorrect. There is no error and no useful warning. On a UK ledger, the guess is wrong most of the time.

**The catch:** Always parse with an explicit `format=` or `dayfirst=True` (both are in the [`pd.to_datetime` documentation](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)). Then check `parsed.min()`, `parsed.max()`, and the month distribution against what the business knows happened.

---

**The pattern across every entry:** The notebook runs, the table renders, and the number is wrong. This week's habits catch it:

- Check `.dtypes` after every read
- Count the rows before and after every operation
- Reconcile against one independent number

## Next week

Week 5 stays in pandas and turns to reshaping data. See the [Week 5 README](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/05/README.md).